In [1]:
%pip install git+https://github.com/illuin-tech/colpali

  Cloning https://github.com/illuin-tech/colpali to /private/var/folders/rh/_g3q29hd09n27f4c0cq7x8kc0000gn/T/pip-req-build-mk_fut33
  Running command git clone --filter=blob:none --quiet https://github.com/illuin-tech/colpali /private/var/folders/rh/_g3q29hd09n27f4c0cq7x8kc0000gn/T/pip-req-build-mk_fut33
  Resolved https://github.com/illuin-tech/colpali to commit b1c47b713ddf7070889347ca3c10cf87dc48c45a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


# Data

In [2]:
import datasets

/opt/anaconda3/envs/gqr-t/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
documents = datasets.load_dataset("vidore/economics_reports_v2",'corpus')["test"]
queries = datasets.load_dataset("vidore/economics_reports_v2",'queries')["test"]
qrels = datasets.load_dataset("vidore/economics_reports_v2",'qrels')["test"]
documents[0]

{'corpus-id': 0,
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=850x1100>,
 'doc-id': 'global-economic-prospects-june-2024'}

In [4]:
from collections import defaultdict
# let's construct a benchmark object, that maps a query to a dict of document_id:relevance_score
benchmark = defaultdict(dict)

for row in qrels:
    qid = row["query-id"]
    did = row["corpus-id"]
    score = row["score"]
    benchmark[qid][did] = score

benchmark = dict(benchmark)
list(benchmark.items())[0]

(0, {63: 2})

# Models

In [5]:
import torch
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = ColIdefics3.from_pretrained(
        "vidore/colSmol-256M",
        dtype=torch.bfloat16,
    ).to(device).eval()
processor = ColIdefics3Processor.from_pretrained("vidore/colSmol-256M")

In [6]:
# Your inputs
images = [row['image'] for row in documents][:10]
queries = [row['query'] for row in queries][:10]

# Process the inputs
batch_images = processor.process_images(images).to(model.device)
batch_queries = processor.process_queries(queries).to(model.device)

# Forward pass
with torch.no_grad():
    image_embeddings = model(**batch_images)
    query_embeddings = model(**batch_queries)

scores = processor.score_multi_vector(query_embeddings, image_embeddings,batch_size=8)
scores

tensor([[4.6875, 4.3125, 4.8438, 7.4688, 7.5312, 7.5000, 7.5000, 7.4688, 7.4688,
         7.4375],
        [4.7188, 4.6250, 4.3125, 6.8438, 6.9062, 6.8438, 6.7188, 6.8438, 6.7188,
         6.5625],
        [3.6562, 4.4375, 5.0625, 4.1562, 4.1875, 4.2812, 4.1875, 4.1562, 4.2500,
         4.2188],
        [5.1562, 4.3125, 4.0625, 6.9375, 6.9062, 7.1875, 7.0625, 6.9375, 7.1875,
         7.0938],
        [5.6875, 6.9062, 6.9375, 7.1562, 7.1875, 7.3750, 7.1875, 7.1562, 7.2188,
         7.1562],
        [3.6875, 5.7188, 5.9062, 8.5625, 8.5625, 8.6875, 8.6250, 8.5625, 8.6250,
         8.5625],
        [4.5938, 5.0938, 5.3438, 7.3438, 7.3438, 7.3125, 7.3125, 7.3438, 7.3125,
         7.3438],
        [4.6562, 6.7188, 6.9062, 7.1875, 7.2188, 7.1875, 7.2188, 7.1875, 7.1875,
         7.2812],
        [4.0625, 4.5938, 4.9062, 7.4062, 7.5000, 7.5625, 7.4375, 7.4062, 7.4375,
         7.4375],
        [3.1406, 4.8750, 5.0312, 6.9062, 6.8750, 7.0000, 6.9062, 6.9062, 6.9375,
         6.9062]])